In [0]:
DROP TABLE IF EXISTS banking.metadata.tables;
DROP TABLE IF EXISTS banking.metadata.table_parameters;
DROP TABLE IF EXISTS banking.metadata.table_watermarks;
DROP TABLE IF EXISTS banking.metadata.pipeline_runs;

In [0]:
CREATE CATALOG IF NOT EXISTS banking;
CREATE SCHEMA IF NOT EXISTS banking.metadata;

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.tables (
    table_id            INT,
    table_name          STRING,
    source_system       STRING,
    source_schema       STRING,
    source_table        STRING,
    source_path         STRING,
    target_layer        STRING,
    bronze_schema       STRING,
    silver_schema       STRING,
    gold_schema         STRING,
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
) USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,
    parameter_value     STRING,
    created_at          TIMESTAMP
) USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.table_watermarks (
    table_id                INT,
    last_watermark_value    STRING,
    last_updated_at         TIMESTAMP,
    last_run_id             BIGINT
) USING DELTA
PARTITIONED BY (table_id);

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.pipeline_runs (
    run_id              BIGINT,
    table_id            INT,
    layer               STRING,
    start_time          TIMESTAMP,
    end_time            TIMESTAMP,
    status              STRING,
    number_of_records   BIGINT,
    error_message       STRING
) USING DELTA
PARTITIONED BY (table_id);

In [0]:
CREATE SCHEMA IF NOT EXISTS banking.source;
CREATE VOLUME IF NOT EXISTS banking.source.volume;

In [0]:
INSERT INTO banking.metadata.tables VALUES
(1, 'customers', 'delta', 'banking', 'customers', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),
(2, 'accounts', 'delta', 'banking', 'accounts', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),
(3, 'transactions', 'delta', 'banking', 'transactions', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),
(4, 'branches', 'delta', 'banking', 'branches', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),
(5, 'credit_bureau_reports', 'blob', NULL, NULL, '/Volumes/banking/source/volume/credit_bureau_reports/', 'silver', 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),
(6, 'payment_gateway_logs', 'blob', NULL, NULL, '/Volumes/banking/source/volume/payment_gateway_logs/', 'silver', 'bronze', 'silver', NULL, TRUE, 6, current_timestamp());

num_affected_rows,num_inserted_rows
6,6


In [0]:
INSERT INTO banking.metadata.table_parameters VALUES
(1, 'load_type', 'MERGE', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
(1, 'watermark_column', 'updated_at', current_timestamp()),
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'account_id', current_timestamp()),
(2, 'watermark_column', 'updated_at', current_timestamp()),
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'txn_id', current_timestamp()),
(3, 'watermark_column', 'txn_timestamp', current_timestamp()),
(4, 'load_type', 'FULL', current_timestamp()),
(4, 'primary_key', 'branch_code', current_timestamp()),
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key', 'customer_id', current_timestamp()),
(5, 'watermark_column', 'bureau_pull_date', current_timestamp()),
(6, 'load_type', 'APPEND', current_timestamp()),
(6, 'primary_key', 'txn_id', current_timestamp()),
(6, 'watermark_column', 'processed_timestamp', current_timestamp());

num_affected_rows,num_inserted_rows
17,17


In [0]:
INSERT INTO banking.metadata.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);

num_affected_rows,num_inserted_rows
5,5


In [0]:
INSERT INTO banking.metadata.tables VALUES
(7, 'customer_360', 'silver', NULL, NULL, NULL, 'gold', NULL, NULL, 'gold', TRUE, 1, current_timestamp()),
(8, 'branch_performance', 'silver', NULL, NULL, NULL, 'gold', NULL, NULL, 'gold', TRUE, 2, current_timestamp()),
(9, 'transaction_channel_summary', 'silver', NULL, NULL, NULL, 'gold', NULL, NULL, 'gold', TRUE, 3, current_timestamp()),
(10, 'daily_bank_kpi', 'silver', NULL, NULL, NULL, 'gold', NULL, NULL, 'gold', TRUE, 4, current_timestamp()),
(11, 'risk_customer_summary', 'silver', NULL, NULL, NULL, 'gold', NULL, NULL, 'gold', TRUE, 1, current_timestamp());

num_affected_rows,num_inserted_rows
5,5


In [0]:
SELECT * FROM banking.metadata.tables;

table_id,table_name,source_system,source_schema,source_table,source_path,target_layer,bronze_schema,silver_schema,gold_schema,active_flag,load_order,created_at
3,transactions,delta,banking,transactions,banking.banking.transactions,silver,bronze,silver,null,true,3,2026-05-12T12:27:25.121Z
1,customers,delta,banking,customers,banking.banking.customers,silver,bronze,silver,null,true,1,2026-05-12T12:27:25.121Z
2,accounts,delta,banking,accounts,banking.banking.accounts,silver,bronze,silver,null,true,2,2026-05-12T12:27:25.121Z
4,branches,delta,banking,branches,banking.banking.branches,silver,bronze,silver,null,true,4,2026-05-12T12:27:25.121Z
5,credit_bureau_reports,blob,null,null,/Volumes/banking/source/volume/credit_bureau_reports/,silver,bronze,silver,null,true,5,2026-05-12T12:27:25.121Z
6,payment_gateway_logs,blob,null,null,/Volumes/banking/source/volume/payment_gateway_logs/,silver,bronze,silver,null,true,6,2026-05-12T12:27:25.121Z
7,customer_360,silver,null,null,null,gold,null,null,gold,true,1,2026-05-12T12:28:04.032Z
8,branch_performance,silver,null,null,null,gold,null,null,gold,true,2,2026-05-12T12:28:04.032Z
9,transaction_channel_summary,silver,null,null,null,gold,null,null,gold,true,3,2026-05-12T12:28:04.032Z
10,daily_bank_kpi,silver,null,null,null,gold,null,null,gold,true,4,2026-05-12T12:28:04.032Z


In [0]:
    SELECT * FROM banking.metadata.table_parameters;

table_id,parameter_name,parameter_value,created_at
1,load_type,MERGE,2026-05-12T12:27:41.290Z
1,primary_key,customer_id,2026-05-12T12:27:41.290Z
1,watermark_column,updated_at,2026-05-12T12:27:41.290Z
2,load_type,MERGE,2026-05-12T12:27:41.290Z
2,primary_key,account_id,2026-05-12T12:27:41.290Z
2,watermark_column,updated_at,2026-05-12T12:27:41.290Z
3,load_type,APPEND,2026-05-12T12:27:41.290Z
3,primary_key,txn_id,2026-05-12T12:27:41.290Z
3,watermark_column,txn_timestamp,2026-05-12T12:27:41.290Z
4,load_type,FULL,2026-05-12T12:27:41.290Z


In [0]:
SELECT * FROM banking.metadata.table_watermarks;

table_id,last_watermark_value,last_updated_at,last_run_id
1,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
2,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
3,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
5,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null
6,1900-01-01 00:00:00,2026-05-12T12:27:53.111Z,null


In [0]:
SELECT table_id, table_name, source_system, source_path 
FROM banking.metadata.tables 
WHERE table_id IN (1,2,3,4)

table_id,table_name,source_system,source_path
3,transactions,delta,banking.banking.transactions
1,customers,delta,banking.banking.customers
2,accounts,delta,banking.banking.accounts
4,branches,delta,banking.banking.branches


In [0]:
UPDATE banking.metadata.tables SET source_system = 'delta', source_path = 'banking.banking.customers' WHERE table_id = 1;
UPDATE banking.metadata.tables SET source_system = 'delta', source_path = 'banking.banking.accounts' WHERE table_id = 2;
UPDATE banking.metadata.tables SET source_system = 'delta', source_path = 'banking.banking.transactions' WHERE table_id = 3;
UPDATE banking.metadata.tables SET source_system = 'delta', source_path = 'banking.banking.branches' WHERE table_id = 4;

num_affected_rows
1


num_affected_rows
1


num_affected_rows
1


num_affected_rows
1
